# Lab 2 — Configurable Dutch cross-language adaptation

**Outcome:** adapt the English Parakeet CTC 0.6B checkpoint to Dutch FLEURS, select the best checkpoint on the official validation split, evaluate once on the untouched test split, and measure English forgetting. Estimated time: 75–100 minutes depending on GPU and the controls below.

> This is a cross-language transfer exercise, not a claim that the English checkpoint is a multilingual production model. The tokenizer remains the base model's English tokenizer, so the notebook normalizes Dutch transcripts to its lower-case Latin alphabet and verifies tokenizer coverage before training.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json, os, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import matplotlib.pyplot as plt
from voice_asr_lab.audio import (
    duration_seconds, load_dummy_librispeech, load_fleurs_records, total_duration,
)
from voice_asr_lab.asr import (
    configure_trainable_parameters, evaluate_wer, fine_tune_with_validation,
    load_model_and_processor, save_trainable_state, tokenizer_coverage,
)
from voice_asr_lab.profiles import detect_profile

## 1. Choose the experiment controls

Edit this one cell to trade runtime for training signal. The defaults keep T4 on the CTC head and use the CTC head plus the last two encoder blocks on L4, A10, and A100. The official FLEURS train, validation, and test splits remain separate regardless of these limits.

In [ ]:
base_profile = detect_profile()

LANGUAGE_CONFIG = 'nl_nl'
LANGUAGE_NAME = 'Dutch'
TRAIN_EXAMPLES = 80 if base_profile.name == 't4' else 200
VALIDATION_EXAMPLES = 20 if base_profile.name == 't4' else 30
TEST_EXAMPLES = 30 if base_profile.name == 't4' else 50
ENGLISH_GUARDRAIL_EXAMPLES = 4
TRAIN_STEPS = 150 if base_profile.name == 't4' else 200
EVAL_EVERY = 25
LEARNING_RATE = 1e-4 if base_profile.name == 't4' else 5e-5
TRAINABLE_ENCODER_LAYERS = 0 if base_profile.name == 't4' else 2
RANDOM_SEED = 7

profile = replace(
    base_profile,
    trainable_encoder_layers=TRAINABLE_ENCODER_LAYERS,
    train_steps=TRAIN_STEPS,
    learning_rate=LEARNING_RATE,
)
assert TRAIN_EXAMPLES > 0 and VALIDATION_EXAMPLES > 0 and TEST_EXAMPLES > 0
assert 0 < EVAL_EVERY <= TRAIN_STEPS
assert TRAINABLE_ENCODER_LAYERS >= -1
print({
    'detected_profile': base_profile.name,
    'language': f'{LANGUAGE_NAME} ({LANGUAGE_CONFIG})',
    'train_examples': TRAIN_EXAMPLES,
    'validation_examples': VALIDATION_EXAMPLES,
    'test_examples': TEST_EXAMPLES,
    'train_steps': profile.train_steps,
    'learning_rate': profile.learning_rate,
    'batch_size': profile.train_batch_size,
    'trainable_encoder_layers': profile.trainable_encoder_layers,
    'max_audio_seconds': profile.max_audio_seconds,
    'approximate_training_passes': round(
        profile.train_steps * profile.train_batch_size / TRAIN_EXAMPLES, 2
    ),
})

## 2. Load duration-safe official splits

FLEURS provides distinct speakers across train, validation, and test. Records longer than the GPU profile allows are filtered before selection. The notebook never truncates audio while retaining a full transcript, because that would create an invalid CTC training pair. The first download is slower; subsequent runs use the Hugging Face cache.

In [ ]:
train_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'train', limit=TRAIN_EXAMPLES,
    max_audio_seconds=profile.max_audio_seconds,
)
validation_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'validation', limit=VALIDATION_EXAMPLES,
    max_audio_seconds=profile.max_audio_seconds,
)
test_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'test', limit=TEST_EXAMPLES,
    max_audio_seconds=profile.max_audio_seconds,
)

english_candidates = load_dummy_librispeech(limit=30)
english_guardrail_records = [
    record for record in english_candidates
    if duration_seconds(record['audio'], record['sampling_rate'])
    <= profile.max_audio_seconds
][:ENGLISH_GUARDRAIL_EXAMPLES]
if len(english_guardrail_records) < ENGLISH_GUARDRAIL_EXAMPLES:
    raise RuntimeError('Not enough duration-safe English guardrail records.')

print({
    'train': {'examples': len(train_records), 'minutes': round(total_duration(train_records) / 60, 1)},
    'validation': {'examples': len(validation_records), 'minutes': round(total_duration(validation_records) / 60, 1)},
    'test': {'examples': len(test_records), 'minutes': round(total_duration(test_records) / 60, 1)},
    'english_guardrail': len(english_guardrail_records),
})
print('Original:  ', train_records[0]['original_text'])
print('Normalized:', train_records[0]['text'])

## 3. Load the English base model and audit tokenizer coverage

The acoustic model can be adapted, but its output vocabulary is fixed. Training stops if any normalized Dutch transcript contains an unknown token. This makes the cross-language constraint visible instead of silently converting unsupported text to `<unk>`.

In [ ]:
model, processor, dtype = load_model_and_processor(training=True)
coverage = tokenizer_coverage(
    processor,
    [record['text'] for record in train_records + validation_records + test_records],
)
print({'dtype': str(dtype), 'tokenizer_coverage': coverage})
if coverage['unknown_tokens']:
    raise RuntimeError(
        'The base tokenizer cannot represent every normalized transcript. '
        'Inspect affected_examples before training.'
    )
parameter_summary = configure_trainable_parameters(model, profile)
parameter_summary

## 4. Freeze the untouched baselines

Validation chooses the checkpoint. Test is reported only for the final comparison and must not influence controls. The small English slice is a forgetting guardrail, not a substitute for a full English regression suite. WER and CER use the same lower-case ASCII normalization as training.

In [ ]:
model.eval()
baseline_validation = evaluate_wer(
    model, processor, validation_records, profile.max_audio_seconds
)
baseline_test = evaluate_wer(
    model, processor, test_records, profile.max_audio_seconds
)
baseline_english = evaluate_wer(
    model, processor, english_guardrail_records, profile.max_audio_seconds
)
print({
    'validation_wer': baseline_validation['wer'],
    'validation_cer': baseline_validation['cer'],
    'test_wer': baseline_test['wer'],
    'test_cer': baseline_test['cer'],
    'english_guardrail_wer': baseline_english['wer'],
})

In [ ]:
for reference, prediction in list(zip(
    baseline_test['references'], baseline_test['predictions']
))[:5]:
    print(f'REF: {reference}\nHYP: {prediction}\n')

## 5. Fine-tune with periodic validation and best-checkpoint selection

Training shuffles the selected train split, clips gradients, evaluates every `EVAL_EVERY` steps, and keeps step 0 as a valid checkpoint. If no validation checkpoint beats the base model, the base trainable parameters are restored instead of publishing a worse final step. Rerun the model-loading cell before starting a new experiment from the base checkpoint.

In [ ]:
training_run = fine_tune_with_validation(
    model,
    processor,
    train_records,
    validation_records,
    profile,
    eval_every=EVAL_EVERY,
    baseline_metrics=baseline_validation,
    seed=RANDOM_SEED,
)
print({
    'best_step': training_run['best_step'],
    'best_validation_wer': training_run['best_wer'],
    'best_validation_cer': training_run['best_cer'],
})

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(training_run['losses']) + 1), training_run['losses'])
axes[0].set(title='Training loss', xlabel='Optimizer step', ylabel='CTC loss')
axes[0].grid(alpha=0.25)
validation_steps = [row['step'] for row in training_run['validation_history']]
axes[1].plot(
    validation_steps,
    [row['wer'] for row in training_run['validation_history']],
    marker='o', label='WER',
)
axes[1].plot(
    validation_steps,
    [row['cer'] for row in training_run['validation_history']],
    marker='o', label='CER',
)
axes[1].axvline(training_run['best_step'], color='black', linestyle='--', alpha=0.5)
axes[1].set(title='Validation metrics', xlabel='Optimizer step', ylabel='Error rate')
axes[1].legend()
axes[1].grid(alpha=0.25)
figure.tight_layout()

## 6. Evaluate the selected checkpoint once on test

The helper has already restored the best validation checkpoint. Test improvement is the honest cross-language result; validation improvement alone is model selection. English WER exposes catastrophic forgetting.

In [ ]:
after_validation = evaluate_wer(
    model, processor, validation_records, profile.max_audio_seconds
)
after_test = evaluate_wer(
    model, processor, test_records, profile.max_audio_seconds
)
after_english = evaluate_wer(
    model, processor, english_guardrail_records, profile.max_audio_seconds
)
checkpoint = save_trainable_state(
    model, ROOT / 'artifacts' / 'lab2_trainable_state.pt'
)
summary = {
    'language': LANGUAGE_NAME,
    'selected_step': training_run['best_step'],
    'baseline_validation_wer': baseline_validation['wer'],
    'selected_validation_wer': after_validation['wer'],
    'baseline_test_wer': baseline_test['wer'],
    'selected_test_wer': after_test['wer'],
    'baseline_test_cer': baseline_test['cer'],
    'selected_test_cer': after_test['cer'],
    'baseline_english_wer': baseline_english['wer'],
    'selected_english_wer': after_english['wer'],
    'checkpoint': str(checkpoint),
}
summary['test_wer_absolute_change'] = (
    summary['selected_test_wer'] - summary['baseline_test_wer']
)
summary

In [ ]:
changed = 0
for reference, before, selected in zip(
    baseline_test['references'], baseline_test['predictions'], after_test['predictions']
):
    if before != selected:
        print(f'REF:    {reference}\nBEFORE: {before}\nAFTER:  {selected}\n')
        changed += 1
    if changed == 5:
        break
if changed == 0:
    print('No decoded test transcript changed at the selected checkpoint.')

In [ ]:
metadata_path = ROOT / 'artifacts' / 'lab2_run_summary.json'
metadata_path.write_text(
    json.dumps({
        **summary,
        'language_config': LANGUAGE_CONFIG,
        'profile': profile.as_dict(),
        'train_examples': len(train_records),
        'validation_examples': len(validation_records),
        'test_examples': len(test_records),
        'eval_every': EVAL_EVERY,
        'random_seed': RANDOM_SEED,
        'tokenizer_coverage': coverage,
        'validation_history': training_run['validation_history'],
    }, indent=2) + '\n',
    encoding='utf-8',
)
print(f'Saved reproducibility metadata to {metadata_path}')

## Interpretation and production boundary

A useful result requires lower error on the untouched Dutch test split without unacceptable English forgetting. A lower training loss or validation WER alone is insufficient. Do not change controls after reading test results; start a new train/validation/test experiment instead.

This checkpoint retains the English base tokenizer and is an educational open-model artifact. It is **not** automatically a supported Dutch Speech NIM package. Lab 3 can export the selected checkpoint through the lower-level ONNX/Triton path, where preprocessing, normalization, tokenizer version, and language behavior must travel with the model.

**Checkpoint:** explain the tokenizer audit, why step 0 can be selected, the validation-versus-test boundary, and the Dutch-improvement/English-forgetting trade-off before continuing to Lab 3.